## Langchain之Chain链

为开发更复杂的应用程序，需要使用Chain来链接LangChain中的各个组件和功能，包括模型之间的链接以及模型与其他组件之间的链接

链在内部把一系列的功能进行封装，而链的外部则又可以组合串联。 链其实可以被视为LangChain中的一种基本功能单元。

### 链的基本使用

LLMChain是最基础也是最常见的链。LLMChain结合了语言模型推理功能，并添加了PromptTemplate和Output Parser等功能，将模型输入输出整合在一个链中操作。

它利用提示模板格式化输入，将格式化后的字符串传递给LLM模型，并返回LLM的输出。这样使得整个处理过程更加高效和便捷。

**未使用Chain**

In [4]:
# 导入LangChain中的提示模板
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()
import os


# 原始字符串模板
template = "桌上有{number}个苹果，四个桃子和 3 本书，一共有几个水果?"

# 创建LangChain模板
prompt_temp = PromptTemplate.from_template(template)

# 根据模板创建提示
prompt = prompt_temp.format(number=2)

from langchain.chat_models import init_chat_model

# 创建模型实例
model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)
# 传入提示，调用模型返回结果
result = model.invoke(prompt)
print(result.content)

我们来看一下桌上的物品：  

- 苹果：2 个（水果）  
- 桃子：4 个（水果）  
- 书：3 本（不是水果）  

题目问的是 **一共有几个水果**，所以只算苹果和桃子：  

\[
2 + 4 = 6
\]

**答：一共有 6 个水果。**


**使用Chain**

In [5]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI

# 原始字符串模板
template = "桌上有{number}个苹果，四个桃子和 3 本书，一共有几个水果?"

# 创建模型实例
llm = init_chat_model(model="deepseek:deepseek-chat")

# 创建LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(template)
)

# 调用LLMChain，返回结果
result = llm_chain.invoke({"number":2})
print(result)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_9572\3133628538.py:12: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


{'number': 2, 'text': '我们先明确题目中的物品：  \n\n- 苹果：2 个（水果）  \n- 桃子：4 个（水果）  \n- 书：3 本（不是水果）  \n\n要求的是 **水果的总数**，所以只算苹果和桃子。  \n\n\\[\n2 + 4 = 6\n\\]\n\n**答案：** 一共有 **6 个水果**。'}


**使用表达式语言 (LCEL)**

LangChain表达式语言，或 LCEL，是一种声明式的方法，可以轻松地将链组合在一起。

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 原始字符串模板
template = "桌上有{number}个苹果，四个桃子和 3 本书，一共有几个水果?"
prompt = PromptTemplate.from_template(template)

# 创建模型实例
llm = init_chat_model(model="deepseek:deepseek-chat")

# 创建Chain  模版 -> 大模型 -> outparse
chain = prompt | llm 
# ps aux 查看所有的进程  | gerp redis  
# chain = llm | prompt

# 调用Chain，返回结果
result = chain.invoke({"number": "3"})
print(result.content)

我们一步步来分析：  

1. 题目说桌上有 **3 个苹果**、**4 个桃子**、**3 本书**。  
2. 问的是 **一共有几个水果**。  
3. 水果包括苹果和桃子，书不是水果。  
4. 水果总数 = 苹果数 + 桃子数 = \( 3 + 4 = 7 \)。  

**答案：7 个水果** ✅


**实战：用LangChain写Python代码并执行来生成答案**

pip install langchain-experimental  

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
)
from langchain_experimental.utilities import PythonREPL

from dotenv import load_dotenv
load_dotenv()

template = """Write some python code to solve the user's problem.

Return only python code in Markdown format, e.g.:

```python
....
```"""
prompt = ChatPromptTemplate.from_messages([("system", template), ("human", "{input}")])

model = init_chat_model(model="deepseek:deepseek-chat")

def _sanitize_output(text: str):
    _, after = text.split("```python")
    return after.split("```")[0] 

# PythonREPL().run 就是调用了一下 exec 函数执行代码
# chain = prompt | model | StrOutputParser()
chain = prompt | model | StrOutputParser() | _sanitize_output | PythonREPL().run
# chain = prompt | model | StrOutputParser() | _sanitize_output
result = chain.invoke({"input": "[1,4,2]给这个列表写一个升序的排序方法"})

print(result)

[1, 2, 4]



### 链的调用方式

**通过invoke方法**

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 原始字符串模板
template = "桌上有{numbers}个苹果，四个桃子和 3 本书，一共有几个水果?"
prompt = PromptTemplate.from_template(template)

# 创建模型实例
llm = init_chat_model(model="deepseek:deepseek-chat")

# 创建Chain
chain = prompt | llm

# 调用Chain，返回结果
result = chain.invoke({"numbers": "3"})
print(result)

content='我们先明确一下题目信息：  \n\n- 苹果：3 个（属于水果）  \n- 桃子：4 个（属于水果）  \n- 书：3 本（不是水果）  \n\n问的是 **一共有几个水果**，所以只算苹果和桃子：  \n\n\\[\n3 + 4 = 7\n\\]\n\n**答案：** 7 个水果 ✅' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 22, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 22}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '34eb9403-6222-4a27-bea3-fbb7105ac6b9', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--1f5073bb-5142-4b6a-8fa8-4d75785c3681-0' usage_metadata={'input_tokens': 22, 'output_tokens': 80, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


**通过batch方法(原apply方法)**:batch方法允许输入列表运行链，一次处理多个输入。

In [15]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 创建模型实例
template = PromptTemplate(
    input_variables=["role", "fruit"],
    template="{role}喜欢吃{fruit}",
)

# 创建LLM
llm = init_chat_model(model="deepseek:deepseek-chat")

# 创建LLMChain
# llm_chain = prompt | llm
llm_chain = LLMChain(llm=llm, prompt=template)

# 输入列表
input_list = [
    {"role": "哪吒", "fruit": "水果"}, {"role": "小猪佩奇", "fruit": "苹果"}
]

# 调用LLMChain，返回结果
result = llm_chain.batch(input_list)
print(result)

[{'role': '哪吒', 'fruit': '水果', 'text': '哪吒喜欢吃水果，这个设定挺有趣的！虽然传统神话故事里没有明确提到哪吒的饮食偏好，但现代改编的影视、动漫或同人创作中，经常赋予他更生活化的特点。如果想象一下：\n\n1. **莲藕与水果的联想**  \n   哪吒重生后用莲藕化身，而莲藕本身是水生植物的根茎，或许他会对清爽多汁的水果（比如莲子、梨、西瓜）感兴趣，毕竟“同源”嘛！\n\n2. **叛逆少年的口味**  \n   哪吒性格桀骜不驯，可能喜欢带点“个性”的水果——比如酸涩的青梅、刺激的榴莲，或者需要剥皮啃咬的芒果，吃相豪迈！\n\n3. **神话的趣味延伸**  \n   如果结合神话元素，也许他偷吃过王母娘娘的蟠桃，或者用火尖枪串糖葫芦（水果版）边走边吃，画面莫名可爱。\n\n4. **现代二创灵感**  \n   在同人作品里，可以设计一些小剧情：哪吒和敖丙逛水果摊，为“荔枝要不要蘸酱油”吵架；或者他用风火轮帮百姓运荔枝，结果自己偷吃半筐……\n\n总之，这个设定让哪吒更接地气，也给了创作更多空间！你是在写故事、画插画，还是单纯脑洞呢？ 😄'}, {'role': '小猪佩奇', 'fruit': '苹果', 'text': '小猪佩奇喜欢吃苹果，这是一个在动画片《小猪佩奇》中常见的设定。佩奇和她的家人、朋友经常一起分享水果，苹果作为健康零食出现，也体现了动画片鼓励健康饮食和家庭分享的理念。  \n\n如果你需要围绕这个主题编故事、设计游戏或制作教育内容，我可以帮你拓展哦！ 😊'}]


**create_stuff_documents_chain：文档链**

create_stuff_documents_chain链将获取文档列表并将它们全部格式化为提示(文档列表)，然后将该提示传递给LLM。

In [18]:
import os
from dotenv import load_dotenv

# 基础依赖
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# 向量库
from langchain_community.vectorstores import FAISS

# Retrieval + Stuff Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


# 环境变量
load_dotenv()


# Step 1: 准备文档
docs = [
    Document(page_content="LangChain 是一个用于构建大语言模型应用的框架。"),
    Document(page_content="LangChain 支持 RAG、Agent、Tool Calling 等能力。"),
    Document(page_content="RAG 的核心思想是：检索外部知识 + 大模型生成。"),
]

# Step 2: 构建向量库
embeddings = DashScopeEmbeddings(model="text-embedding-v3")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# Step 3: 定义 Prompt（⚠️ 必须有 {context}）
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "你是一个严谨的 AI 助手，请仅基于以下文档内容回答问题。\n\n"
     "文档内容：{context}"
    ),
    ("human", "{input}")
])


# Step 4: 初始化 LLM
llm = init_chat_model(model="deepseek:deepseek-chat")


# Step 5: 创建 Stuff Documents Chain
combine_docs_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

# Step 6: 创建 Retrieval Chain（RAG）
rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=combine_docs_chain
)


# Step 7: 调用
result = rag_chain.invoke({
    "input": "什么是 RAG？LangChain 在其中起什么作用？"
})

print("====== 最终回答 ======")
print(result["answer"])

====== 最终回答 ======
根据文档内容：

**RAG** 的核心思想是：检索外部知识 + 大模型生成。

**LangChain** 在其中起的作用是：作为一个支持 RAG 能力的框架，用于构建大语言模型应用。


**LLMMathChain：数学链**

LLMMathChain将用户问题转换为数学问题，然后将数学问题转换为可以使用 Python 的 numexpr 库执行的表达式。使用运行此代码的输出来回答问题

使用LLMMathChain，需要安装numexpr库

pip install numexpr

In [19]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMMathChain

# 初始化大模型
llm = init_chat_model(model="deepseek:deepseek-chat")

# 创建链
llm_math = LLMMathChain.from_llm(llm)

# 执行链
res = llm_math.invoke("10 ** 3 + 100的结果是多少？")
# res = llm.invoke("10 ** 3 + 100的结果是多少？")
print(res)

{'question': '10 ** 3 + 100的结果是多少？', 'answer': 'Answer: 1100'}


**create_sql_query_chain：SQL查询链**

create_sql_query_chain是创建生成SQL查询的链，用于将自然语言转换成数据库的SQL查询

这里使用MySQL数据库，需要安装pymysql

pip install pymysql

In [20]:
from langchain_community.utilities import SQLDatabase

# 连接 sqlite 数据库
# db = SQLDatabase.from_uri("sqlite:///demo.db")  

# 连接 MySQL 数据库
db_user = "root"
db_password = "root"
db_host = "127.0.0.1"
db_port = "3306"
db_name = "func"
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

print("那种数据库：",db.dialect)
print("获取数据表：",db.get_usable_table_names())
# 执行查询
res = db.run("SELECT count(*) FROM students;")
print("查询结果：",res)

那种数据库： mysql
获取数据表： ['classes', 'goods', 'scores', 'students']
查询结果： [(5,)]


In [24]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
# 初始化大模型
llm = init_chat_model(model="deepseek:deepseek-chat")

chain = create_sql_query_chain(llm=llm, db=db)
# user = input("输入要查询的需求:")

# response = chain.invoke({"question": "数据表orders哪个用户消费最高？"})
# response = chain.invoke({"question": "查询商品分类表中的商品分类有多少种？只返回可以执行的SQL语句"})
# response = chain.invoke({"question": "查询一班的学生数学成绩是多少？只返回可以执行的SQL语句"})
# response = chain.invoke({"question": user})
# 限制使用的表  
# response = chain.invoke({"question": "查询一班的学生数学成绩是多少？只返回可以执行的SQL语句", "table_names_to_use": ["students"]})
print(response)

def _sanitize_output(text: str):
    _, after = text.split("```sql")
    return after.split("```")[0]

result = _sanitize_output(response)
res = db.run(result)
print("查询结果：",res)

```sql
SELECT `name`, `score` FROM `students` 
WHERE `class_id` = 1 AND `subject` = '数学'
LIMIT 5;
```


OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'score' in 'field list'")
[SQL: 
SELECT `name`, `score` FROM `students` 
WHERE `class_id` = 1 AND `subject` = '数学'
LIMIT 5;
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)